Environment Setup and Data Load

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
import pickle
from sklearn.preprocessing import LabelEncoder

#drive mount
print("Mounting drive...")
drive.mount('/content/drive')

base_path = '/content/drive/My Drive/resources/KuaiRec 2.0/data/'

print("Loading data...")
big_matrix = pd.read_csv(base_path + 'big_matrix.csv')
item_categories = pd.read_csv(base_path + 'item_categories.csv')
print("Data loaded.")

#sample view
print("\n Big matrix")
print(big_matrix.head(3))
print("\n item categories")
print(item_categories.head(3))

Mounting drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading data...
Data loaded.

 Big matrix
   user_id  video_id  play_duration  video_duration                     time  \
0        0      3649          13838           10867  2020-07-05 00:08:23.438   
1        0      9598          13665           10984  2020-07-05 00:13:41.297   
2        0      5262            851            7908  2020-07-05 00:16:06.687   

       date     timestamp  watch_ratio  
0  20200705  1.593879e+09     1.273397  
1  20200705  1.593879e+09     1.244082  
2  20200705  1.593879e+09     0.107613  

 item categories
   video_id     feat
0         0      [8]
1         1  [27, 9]
2         2      [9]


In [ ]:
#merging the first item in categories into the matrix
item_categories['category_id'] = item_categories['feat'].apply(lambda x: eval(x)[0])

clean_categories = item_categories[['video_id', 'category_id']]

print("Merging tables....")
df_merged = big_matrix.merge(clean_categories, on='video_id', how = 'left')

#sample view
print(df_merged[['user_id', 'video_id', 'watch_ratio', 'time', 'category_id']].head(5))

Merging tables....
   user_id  video_id  watch_ratio                     time  category_id
0        0      3649     1.273397  2020-07-05 00:08:23.438            9
1        0      9598     1.244082  2020-07-05 00:13:41.297           28
2        0      5262     0.107613  2020-07-05 00:16:06.687           25
3        0      1963     0.089885  2020-07-05 00:20:26.792            9
4        0      8234     0.078000  2020-07-05 00:43:05.128            6


Implicit Feedback

In [ ]:
#filtering only the positive interactions
df_clean = df_merged[df_merged['watch_ratio'] >= 1.0].copy()

#sample view
print(f"min watch_ratio: {df_clean['watch_ratio'].min()}")
print(f"max watch_ratio: {df_clean['watch_ratio'].max()}")
print(f"remaning rows: {len(df_clean)}")

min watch_ratio: 1.0
max watch_ratio: 573.4571428571429
remaning rows: 4238228


Remapping for Improvement

In [ ]:
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
cat_encoder = LabelEncoder()

df_clean['user_id'] = user_encoder.fit_transform(df_clean['user_id'])
df_clean['video_id'] = item_encoder.fit_transform(df_clean['video_id'])
df_clean['category_id'] = cat_encoder.fit_transform(df_clean['category_id'].fillna(-1).astype(str))

num_users = len(user_encoder.classes_)
num_items = len(item_encoder.classes_)
num_cats = len(cat_encoder.classes_)
print(f"Dataset remapped: {num_users} users, {num_items} items, {num_cats} categories")

Dataset remapped: 7176 users, 10719 items, 31 categories


Sequences

In [ ]:
#sort by chronological order
df_clean.sort_values(by = ['user_id', 'time'], inplace = True)

sequences = df_clean.groupby('user_id').agg({
    'video_id': list,
    'category_id': list})
sequences.columns = ['item_sequence', 'category_sequence']

#sample view
print(sequences.head(5))

                                             item_sequence  \
user_id                                                      
0        [3647, 9590, 8221, 6807, 169, 1987, 5271, 179,...   
1        [5272, 3647, 183, 9586, 9562, 1978, 148, 5216,...   
2        [171, 6720, 214, 211, 3682, 8266, 8254, 9619, ...   
3        [5249, 1902, 8210, 9593, 8121, 6798, 5241, 819...   
4        [6796, 1974, 5271, 8123, 1985, 1987, 128, 5229...   

                                         category_sequence  
user_id                                                     
0        [30, 21, 1, 11, 3, 21, 18, 21, 27, 30, 18, 21,...  
1        [21, 30, 21, 8, 21, 11, 3, 25, 20, 21, 7, 11, ...  
2        [18, 18, 4, 21, 21, 5, 30, 7, 30, 25, 21, 19, ...  
3        [3, 26, 3, 13, 23, 21, 13, 29, 18, 18, 7, 29, ...  
4        [30, 2, 18, 3, 19, 21, 3, 19, 9, 2, 9, 30, 30,...  


Train/Test Split

In [ ]:
#users with at least 2 items
sequences[sequences['item_sequence'].apply(len) >= 2].copy()

test_targets_series = sequences['item_sequence'].apply(lambda x: x[-1])

#for training sequence, modifying the df
sequences['item_sequence'] = sequences['item_sequence'].apply(lambda x: x[:-1])
sequences['category_sequence'] = sequences['category_sequence'].apply(lambda x: x[:-1])

#dict conversion
train_sequences = sequences.to_dict(orient = 'index')
test_targets  = test_targets_series.to_dict()

print(f"split completed")

#sample view
print(f"training set users: {len(train_sequences)}")
print(f"test set users: {len(test_targets)}")

split completed
training set users: 7176
test set users: 7176


Normalized Popularity Weight Calculation (IPW)

In [ ]:
#column list to vertical series
train_items_series = sequences['item_sequence'].explode()

#counting interactions
item_counts = train_items_series.value_counts()
total_train_interactions = len(train_items_series)

#probability calculation
probs = item_counts / total_train_interactions

#inverse propensity with 0.5 smoothing
weight_series = 1.0 / (probs ** 0.5)

#normalize
weight_series = weight_series / weight_series.mean()

#dict conversion
ipw_weights = weight_series.to_dict()

print(f"normalized ipw calculated")

#sample view
print(f"unique items scored: {len(ipw_weights)}")
#avg should be 1.0
print(f"avg weight: {weight_series.mean():.4f}")
print(f"max weight/niche item: {weight_series.max():.4f}")
print(f"min weight/viral item: {weight_series.min():.4f}")

normalized ipw calculated
unique items scored: 10706
avg weight: 1.0000
max weight/niche item: 4.3865
min weight/viral item: 0.0637


Save Checkpoint

In [ ]:
import os

print("saving files..")
new_folder_path = base_path + 'processed/'

if not os.path.exists(new_folder_path):
  os.makedirs(new_folder_path)
  print(f"created a new folder {new_folder_path}")
else:
  print(f"saving to an existing folder: {new_folder_path}")

#saving key objects for other notebooks

meta = {
    'num_users': num_users,
    'num_items': num_items,
    'num_cats': num_cats
}

with open(new_folder_path + 'train_sequence.pkl', 'wb') as f:
  pickle.dump(train_sequences, f)

with open(new_folder_path + 'test_targets.pkl', 'wb') as f:
    pickle.dump(test_targets, f)

with open(new_folder_path + 'ipw_weights.pkl', 'wb') as f:
    pickle.dump(ipw_weights, f)

with open(new_folder_path + 'meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print("all files saved.")

saving files..
saving to an existing folder: /content/drive/My Drive/resources/KuaiRec 2.0/data/processed/
all files saved.
